In [1]:
"""
Expedia downloader (shadow DOM + iframes)
- Login if needed
- Open gear -> "Download"
- In modal: Format=CSV, set filename (YYYY-MM), Unformatted + All results
- Click "Download" robustly (multi-fallback + allow multiple downloads)
- Wait up to 30 minutes with 1-minute ticks while monitoring download folder
"""

import os
import time
from datetime import datetime
from urllib.parse import urlparse

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

# Optional: webdriver-manager
try:
    from webdriver_manager.chrome import ChromeDriverManager
    HAS_WDM = True
except Exception:
    HAS_WDM = False


# ---------- Config ----------
expedia_url = "https://console.vap.expedia.com/analytics-console-user-interface/reports/personal_reports/t3_cnx"
path_fragment = urlparse(expedia_url).path.split("/")[-1]

PROFILE_DIR = r"C:/temp/new_chrome_profile"
CHROMEDRIVER = r"C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/CAPTURE/chromedriver-win64/chromedriver.exe"
DOWNLOAD_DIR = r"C:/temp/expedia_downloads"

ACTION_DELAY = 1                    # delay per action (s)
PAGE_LOAD_TIMEOUT = 20                # page load wait (s)
DEFAULT_WAIT = 30
WAIT_AFTER_OPEN = 20                  # seconds: hard wait after page is fully ready

NAME_PREFIX = "New Look Excel data_EN"
PERIOD = datetime.now().strftime("%Y-%m")  # e.g. '2025-10'

# Wait/monitor config
TICK_SECONDS = 60           # mỗi lần kiểm tra 1 phút
MAX_WAIT_MINUTES = 30       # tối đa 30 phút
EXT_DONE = (".csv", ".zip", ".xlsx")  # các đuôi có thể trả về

# SVG 'gear' path prefix
GEAR_PATH_PREFIX = (
    "M19.43 12.98c.04-.32.07-.64.07-.98 0-.34-.03-.66-.07-.98l2.11-1.65c.19-.15"
    ".24-.42.12-.64l-2-3.46a.5.5 0 00-.61-.22l-2.49 1c-.52-.4-1.08-.73-1.69-.98l-."
    "38-2.65A.488.488 0 0014 2h-4c-.25 0-.46.18-.49.42l-.38 2.65c-.61.25-1.17.59-1."
    "69.98l-2.49-1a.566.566 0 00-.18-.03c-.17 0-.34.09-.43.25l-2 3.46c-.13.22-.07."
    "49.12.64l2.11 1.65c-.04.32-.07.65-.07.98 0 .33.03.66.07.98l-2.11 1.65c-.19.15-."
    "24.42-.12.64l2 3.46a.5.5 0 00.61.22l2.49-1c.52.4 1.08.73 1.69.98l.38 2.65c.03."
    "24.24.42.49.42h4c.25 0 .46-.18.49-.42l.38-2.65c.61-.25 1.17-.59 1.69-.98l2.49 1c."
    "06.02.12.03.18.03.17 0 .34-.09.43-.25l2-3.46c.12-.22.07-.49-.12-.64l-2.11-1.65zm-"
    "1.98-1.71c.04.31.05.52.05.73 0 .21-.02.43-.05.73l-.14 1.13.89.7 1.08.84-.7 1.21-"
    "1.27-.51-1.04-.42-.9.68c-.43.32-.84.56-1.25.73l-1.06.43-.16 1.13-.2 1.35h-1.4l-."
    "19-1.35-.16-1.13-1.06-.43c-.43-.18-.83-.41-1.23-.71l-.91-.7-1.06.43-1.27.51-.7-"
    "1.21 1.08-.84.89-.7-.14-1.13c-.03-.31-.05-.54-.05-.74s.02-.43.05-.73l.14-1.13-."
    "89-.7-1.08-.84.7-1.21 1.27.51 1.04.42.9-.68c.43-.32.84-.56 1.25-.73l1.06-.43.16-"
    "1.13.2-1.35h1.39l.19 1.35.16 1.13 1.06.43c.43.18.83.41 1.23.71l.91.7 1.06-.43 1.27-"
    ".51.7 1.21-1.07.85-.89.7.14 1.13zM12 8c-2.21 0-4 1.79-4 4s1.79 4 4 4 4-1.79 4-4-1."
    "79-4-4-4zm0 6c-1.1 0-2-.9-2-2s.9-2 2-2 2 .9 2 2-.9 2-2 2z"
)


def pause():
    time.sleep(ACTION_DELAY)


# ---------- Setup ----------
def build_driver():
    os.makedirs(DOWNLOAD_DIR, exist_ok=True)

    o = Options()
    o.add_argument(f"--user-data-dir={PROFILE_DIR}")
    o.add_argument("--profile-directory=Default")
    o.add_argument("--start-maximized")
    o.add_experimental_option(
        "prefs",
        {
            "download.default_directory": DOWNLOAD_DIR,
            "download.prompt_for_download": False,
            "download.directory_upgrade": True,
            "safebrowsing.enabled": True,
            "profile.default_content_setting_values.automatic_downloads": 1,  # allow multiple
        },
    )

    if os.path.exists(CHROMEDRIVER):
        print("Using local ChromeDriver path.")
        driver = webdriver.Chrome(service=Service(CHROMEDRIVER), options=o)
    elif HAS_WDM:
        print("Using WebDriverManager to resolve ChromeDriver.")
        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=o)
    else:
        print("Using Selenium Manager (built-in).")
        driver = webdriver.Chrome(options=o)

    driver.set_page_load_timeout(PAGE_LOAD_TIMEOUT)
    driver.set_script_timeout(180)

    # Allow downloads via CDP (best-effort)
    try:
        driver.execute_cdp_cmd(
            "Page.setDownloadBehavior",
            {"behavior": "allow", "downloadPath": DOWNLOAD_DIR},
        )
    except Exception:
        pass

    return driver


def check_and_login(driver, url, timeout=PAGE_LOAD_TIMEOUT):
    driver.get(url)
    WebDriverWait(driver, timeout).until(lambda d: d.execute_script("return document.readyState") == "complete")
    print("Page loaded.")
    pause()

    # Okta sign-in if needed
    try:
        btn = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, 'button[data-testid="console-okta-sign-in"]'))
        )
        print("Sign-in required detected. Clicking sign-in...")
        btn.click()
        pause()

        try:
            lab = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, 'label[for="input36"][data-se-for-name="rememberMe"]'))
            )
            lab.click()
            pause()
        except TimeoutException:
            pass

        try:
            nxt = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable(
                    (By.CSS_SELECTOR, 'input.button.button-primary[type="submit"][value="Next"]')
                )
            )
            nxt.click()
            print("Clicked 'Next'.")
            pause()
        except TimeoutException:
            pass

        try:
            WebDriverWait(driver, 5).until(EC.alert_is_present())
            driver.switch_to.alert.accept()
            print("Accepted alert.")
            pause()
        except Exception:
            pass

        driver.get(url)
        print("Login successful. Reloading target page...")
        pause()
    except TimeoutException:
        print("No sign-in required.")

    # Ensure final target page is fully ready
    WebDriverWait(driver, PAGE_LOAD_TIMEOUT).until(EC.url_contains(path_fragment))
    WebDriverWait(driver, PAGE_LOAD_TIMEOUT).until(lambda d: d.execute_script("return document.readyState") == "complete")
    print("Page ready.")

    # Hard wait after opening page
    print(f"Hard-waiting {WAIT_AFTER_OPEN}s before interacting with gear...")
    time.sleep(WAIT_AFTER_OPEN)
    print("Hard-wait finished.")
    pause()


# ---------- Deep utilities (shadow + iframes) ----------
_DEEP_QUERY_JS = r"""
const sel = arguments[0];
function deepQuery(root) {
  const el = root.querySelector(sel);
  if (el) return el;
  const all = root.querySelectorAll('*');
  for (const n of all) if (n.shadowRoot) {
    const f = deepQuery(n.shadowRoot);
    if (f) return f;
  }
  return null;
}
return deepQuery(document);
"""

def _for_all_contexts(driver, fn):
    if fn():
        return True
    frames = driver.find_elements(By.TAG_NAME, "iframe")
    for fr in frames:
        try:
            driver.switch_to.frame(fr)
            if _for_all_contexts(driver, fn):
                return True
            driver.switch_to.parent_frame()
        except Exception:
            try:
                driver.switch_to.parent_frame()
            except Exception:
                pass
    return False


def deep_try(driver, fn, timeout=20, poll=0.3):
    end = time.time() + timeout
    while time.time() < end:
        driver.switch_to.default_content()
        if _for_all_contexts(driver, fn):
            return True
        time.sleep(poll)
        driver.switch_to.default_content()
    return False


def deep_get(driver, css, timeout=20):
    holder = {"el": None}

    def fn():
        el = driver.execute_script(_DEEP_QUERY_JS, css)
        if el:
            holder["el"] = el
            return True
        return False

    if not deep_try(driver, fn, timeout):
        raise TimeoutException(f"Deep element not found: {css}")
    return holder["el"]


def deep_click_text(driver, text, timeout=15):
    js = r"""
const target = arguments[0];
function deepFindByText(root) {
  const all = root.querySelectorAll('*');
  for (const el of all) {
    if (el.shadowRoot) {
      const f = deepFindByText(el.shadowRoot);
      if (f) return f;
    }
    const t = (el.innerText || el.textContent || '').trim();
    if (!t) continue;
    if (t === target) {
      return el.closest('[role="option"],li,button,[role="button"],a,div,span') || el;
    }
  }
  return null;
}
return deepFindByText(document);
"""
    holder = {"el": None}

    def fn():
        el = driver.execute_script(js, text)
        if el:
            holder["el"] = el
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", el)
            try:
                el.click()
            except Exception:
                driver.execute_script("arguments[0].click();", el)
            return True
        return False

    if not deep_try(driver, fn, timeout):
        raise TimeoutException(f"Deep text not found: {text}")
    pause()
    return holder["el"]


def deep_find_and_switch(driver, css, timeout=20):
    """Return element and keep driver INSIDE its iframe."""
    end = time.time() + timeout

    def search_here():
        el = driver.execute_script(_DEEP_QUERY_JS, css)
        if el:
            return el
        frames = driver.find_elements(By.TAG_NAME, "iframe")
        for fr in frames:
            try:
                driver.switch_to.frame(fr)
                found = search_here()
                if found:
                    return found
                driver.switch_to.parent_frame()
            except Exception:
                try:
                    driver.switch_to.parent_frame()
                except Exception:
                    pass
        return None

    while time.time() < end:
        driver.switch_to.default_content()
        el = search_here()
        if el:
            return el
        time.sleep(0.3)
        driver.switch_to.default_content()

    raise TimeoutException(f"Deep element not found: {css}")


# ---------- Actions ----------
def click_gear_by_path(driver, d_prefix=GEAR_PATH_PREFIX, timeout=25):
    js = r"""
const dPrefix = arguments[0];
function deepWalk(root) {
  const paths = root.querySelectorAll('path[d]');
  for (const p of paths) {
    const d = p.getAttribute('d') || '';
    if (d && (d === dPrefix || d.startsWith(dPrefix))) {
      return p.closest('button,[role="button"],a,div,span') || p;
    }
  }
  const all = root.querySelectorAll('*');
  for (const el of all) {
    if (el.shadowRoot) {
      const f = deepWalk(el.shadowRoot);
      if (f) return f;
    }
  }
  return null;
}
return deepWalk(document);
"""
    holder = {"el": None}

    def fn():
        el = driver.execute_script(js, d_prefix)
        if el:
            holder["el"] = el
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", el)
            try:
                el.click()
            except Exception:
                driver.execute_script("arguments[0].click();", el)
            return True
        return False

    if not deep_try(driver, fn, timeout):
        raise TimeoutException("Gear path not found across frames/shadow DOM.")

    print("Clicked gear.")
    pause()


def open_download_modal(driver):
    try:
        deep_click_text(driver, "Download", timeout=6)
        print("Opened Download directly.")
        return
    except TimeoutException:
        pass

    click_gear_by_path(driver)
    deep_click_text(driver, "Download", timeout=10)
    print("Opened Download via gear.")


def set_download_format_to_csv(driver):
    btn = deep_get(driver, "#listbox-input-qr-export-modal-format", timeout=20)
    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
    try:
        btn.click()
    except Exception:
        driver.execute_script("arguments[0].click();", btn)
    pause()

    try:
        deep_click_text(driver, "CSV", timeout=8)
        print("Selected CSV.")
    except TimeoutException:
        btn.send_keys("CSV")
        btn.send_keys(Keys.ENTER)
        pause()
        print("Entered CSV via keyboard.")

    btn2 = deep_get(driver, "#listbox-input-qr-export-modal-format", timeout=10)
    if (btn2.get_attribute("value") or "").strip().upper() != "CSV":
        raise TimeoutException("Format not set to CSV.")
    print("Format is now CSV.")
    pause()


def set_export_filename(driver, name_prefix=NAME_PREFIX, period=PERIOD):
    box = deep_get(driver, "#qr-export-modal-custom-filename", timeout=20)
    filename = f"{name_prefix} {period}"
    try:
        box.click()
        pause()
        box.send_keys(Keys.CONTROL, "a")
        pause()
        box.send_keys(Keys.BACK_SPACE)
        pause()
        box.send_keys(filename)
        pause()
    except Exception:
        driver.execute_script(
            """
const el = arguments[0], val = arguments[1];
const setter = Object.getOwnPropertyDescriptor(HTMLInputElement.prototype, 'value').set;
setter.call(el, val);
el.dispatchEvent(new Event('input', {bubbles:true}));
el.dispatchEvent(new Event('change', {bubbles:true}));
""",
            box,
            filename,
        )
        pause()

    if (box.get_attribute("value") or "").strip() != filename:
        raise TimeoutException("Filename not applied.")
    print(f"Set filename to: {filename}")
    pause()


def select_unformatted_and_all_results(driver):
    deep_click_text(driver, "Unformatted (no rounding, special characters, etc.)", timeout=15)
    pause()
    deep_click_text(driver, "All results", timeout=10)
    pause()
    # Chờ nút Download thực sự enable sau khi UI re-render
    _deep_find_enabled_download(driver, timeout=20)
    print("Selected Unformatted + All results.")
    pause()


# ---------- Robust Download click ----------
def _absolute_center_in_page(driver, el):
    return driver.execute_script(
        """
const el = arguments[0];
const r = el.getBoundingClientRect();
let x = (r.left + r.right) / 2;
let y = (r.top + r.bottom) / 2;
let w = window;
for (let i = 0; i < 15; i++) {
  if (w === w.parent) break;
  try {
    const fr = w.frameElement;
    if (!fr) break;
    const rf = fr.getBoundingClientRect();
    x += rf.left;
    y += rf.top;
    w = w.parent;
  } catch (e) { break; }
}
return {x, y};
""",
        el,
    )


def _cdp_click_at(driver, x, y):
    driver.execute_cdp_cmd("Input.dispatchMouseEvent", {"type": "mouseMoved", "x": x, "y": y, "buttons": 1})
    driver.execute_cdp_cmd(
        "Input.dispatchMouseEvent", {"type": "mousePressed", "x": x, "y": y, "button": "left", "buttons": 1, "clickCount": 1}
    )
    driver.execute_cdp_cmd(
        "Input.dispatchMouseEvent", {"type": "mouseReleased", "x": x, "y": y, "button": "left", "buttons": 1, "clickCount": 1}
    )


def _dispatch_pointer_click(driver, el):
    driver.execute_script("""
      const el = arguments[0];
      el.scrollIntoView({block:'center'});
      const opts = {bubbles:true, cancelable:true, composed:true};
      el.dispatchEvent(new PointerEvent('pointerover', opts));
      el.dispatchEvent(new PointerEvent('pointerenter', opts));
      el.dispatchEvent(new MouseEvent('mouseover', opts));
      el.dispatchEvent(new PointerEvent('pointerdown', opts));
      el.dispatchEvent(new MouseEvent('mousedown', opts));
      el.dispatchEvent(new MouseEvent('click', opts));
      el.dispatchEvent(new MouseEvent('mouseup', opts));
      el.dispatchEvent(new PointerEvent('pointerup', opts));
    """, el)


def _deep_find_enabled_download(driver, timeout=30, poll=0.2):
    """Return (btn, True) when Download is enabled and keep inside its iframe."""
    end = time.time() + timeout
    last_err = None
    while time.time() < end:
        try:
            driver.switch_to.default_content()
            btn = deep_find_and_switch(driver, "button#qr-export-modal-download, #qr-export-modal-download", timeout=5)
            dis = (btn.get_attribute("disabled") or "").strip().lower()
            aria = (btn.get_attribute("aria-disabled") or "").strip().lower()
            if btn.is_displayed() and dis in ("", "false", None) and aria in ("", "false", None):
                return btn, True
        except Exception as e:
            last_err = e
        time.sleep(poll)
    raise TimeoutException(f"Download button not enabled in time: {last_err}")


def _collect_download_candidates_in_iframe(driver):
    candidates = driver.execute_script("""
      function visible(el){
        const r = el.getBoundingClientRect();
        const cs = getComputedStyle(el);
        return r.width>0 && r.height>0 && cs.visibility!=='hidden' && cs.display!=='none';
      }
      function textOk(el){
        const t = (el.innerText || el.textContent || '').trim();
        return t === 'Download';
      }
      const out = [];
      const all = document.querySelectorAll('button,[role="button"],a,div');
      for (const el of all){
        if (!visible(el)) continue;
        if (!textOk(el)) continue;
        const cs = getComputedStyle(el);
        let zi = 0;
        const z = cs.zIndex;
        if (z && !isNaN(+z)) zi = +z;
        out.push({el, zi, cls: el.className || ''});
      }
      const seen = new Set();
      const uniq = [];
      for (const it of out){
        if (!seen.has(it.el)) { seen.add(it.el); uniq.push(it); }
      }
      uniq.sort((a,b) => (b.zi - a.zi) || (b.el.compareDocumentPosition(a.el) & Node.DOCUMENT_POSITION_FOLLOWING ? 1 : -1));
      return uniq;
    """)
    py = []
    for it in candidates or []:
        py.append({"el": it["el"], "z": it["zi"], "cls": it["cls"]})
    print(f"[DEBUG] Found {len(py)} 'Download' candidates in modal.")
    for i, it in enumerate(py):
        print(f"  - cand#{i+1}: z={it['z']} class='{it['cls'][:120]}'")
    return py


def click_modal_download(driver):
    """
    Robustly click Download:
    - Re-find button before every action to avoid stale refs
    - Enable + focus + center + hover
    - Temporarily disable any overlay at absolute click point (top document)
    - Click via Actions / JS / CDP at center and a few offsets
    - Stop early if .crdownload appears
    """
    from selenium.common.exceptions import (
        StaleElementReferenceException,
        NoSuchElementException,
        JavascriptException,
    )

    def _find_btn(timeout=8):
        driver.switch_to.default_content()
        return deep_find_and_switch(
            driver, "button#qr-export-modal-download, #qr-export-modal-download", timeout=timeout
        )

    def _has_started_download():
        try:
            pref = f"{NAME_PREFIX} {PERIOD}".lower()
            for n in os.listdir(DOWNLOAD_DIR):
                if n.lower().startswith(pref) and n.endswith(".crdownload"):
                    return True
        except Exception:
            pass
        return False

    def _prep(btn_el):
        try:
            driver.execute_script("""
                const b = arguments[0];
                b.removeAttribute('disabled');
                b.setAttribute('aria-disabled','false');
                b.scrollIntoView({block:'center'});
                try { b.focus({preventScroll:true}); } catch(e) {}
            """, btn_el)
        except Exception:
            pass

    def _abs_center_retry(btn_el, attempts=3):
        """Tính tọa độ tuyệt đối, tự re-find nếu stale/no-such."""
        last_err = None
        for _ in range(attempts):
            try:
                return _absolute_center_in_page(driver, btn_el)
            except (StaleElementReferenceException, NoSuchElementException, JavascriptException) as e:
                last_err = e
                # re-find rồi thử lại
                try:
                    btn_el = _find_btn(timeout=5)
                except Exception:
                    time.sleep(0.2)
        # lần cuối cùng vẫn fail -> ném lỗi gốc (để workflow biết)
        if last_err:
            raise last_err
        return {"x": 0, "y": 0}

    def _hover_abs(absx, absy):
        try:
            driver.execute_cdp_cmd("Input.dispatchMouseEvent",
                                   {"type": "mouseMoved", "x": float(absx), "y": float(absy), "buttons": 1})
        except Exception:
            pass

    def _clear_overlay_at(absx, absy, restore_ms=800):
        # Tắt tạm pointer-events của phần tử top tại điểm click
        try:
            driver.switch_to.default_content()
            driver.execute_script("""
                (function(x,y,ms){
                  const el = document.elementFromPoint(x,y);
                  if(!el) return;
                  const old = el.style.pointerEvents;
                  el.style.pointerEvents = 'none';
                  setTimeout(()=>{ el.style.pointerEvents = old; }, ms);
                })(arguments[0], arguments[1], arguments[2]);
            """, float(absx), float(absy), int(restore_ms))
        except Exception:
            pass
        # quay lại iframe modal
        try:
            deep_find_and_switch(driver, "button#qr-export-modal-download, #qr-export-modal-download", timeout=2)
        except Exception:
            pass

    # ---- rounds ----
    rounds = 5
    for r in range(rounds):
        # luôn re-find để tránh stale giữa các vòng
        try:
            btn = _find_btn(timeout=10)
        except Exception:
            time.sleep(0.2)
            continue

        _prep(btn)

        # Tính tọa độ tuyệt đối với retry an toàn
        coords = _abs_center_retry(btn, attempts=3)
        absx, absy = float(coords["x"]), float(coords["y"])

        # Tắt overlay & hover để kích hoạt 'bg-on'
        _clear_overlay_at(absx, absy)
        _hover_abs(absx, absy)
        time.sleep(0.08)

        # 1) Actions click
        try:
            btn = _find_btn(timeout=4)      # re-find trước khi click
            ActionChains(driver).move_to_element(btn).pause(0.05).click().perform()
        except Exception:
            pass
        time.sleep(0.12)
        if _has_started_download():
            print("Clicked Download → detected .crdownload after Actions.")
            return time.time()

        # 2) JS click lên element hiện tại
        try:
            btn = _find_btn(timeout=4)
            driver.execute_script("arguments[0].click();", btn)
        except Exception:
            pass
        time.sleep(0.15)
        if _has_started_download():
            print("Clicked Download → detected .crdownload after el.click().")
            return time.time()

        # 3) CDP click theo tọa độ (tâm + offsets)
        try:
            driver.switch_to.default_content()
            for dx, dy in [(0,0), (3,2), (-3,-2), (2,-3), (-2,3)]:
                _cdp_click_at(driver, absx + dx, absy + dy)
                time.sleep(0.08)
        except Exception:
            pass
        # trở lại iframe cho vòng kế
        try:
            deep_find_and_switch(driver, "button#qr-export-modal-download, #qr-export-modal-download", timeout=2)
        except Exception:
            pass

        if _has_started_download():
            print("Clicked Download → detected .crdownload after CDP.")
            return time.time()

        time.sleep(0.6)  # nghỉ ngắn, đề phòng UI re-render

    print("Clicked Download (best-effort).")
    return time.time()

# ---------- Download monitoring (1-minute ticks, max 30 minutes) ----------
def _human_size(n):
    try:
        n = float(n)
    except Exception:
        return "?"
    for unit in ("B","KB","MB","GB","TB"):
        if n < 1024.0:
            return f"{n:.1f}{unit}"
        n /= 1024.0
    return f"{n:.1f}PB"


def _list_matching_files(dirpath, base_prefix, since_ts=None):
    """Trả về các file bắt đầu bằng base_prefix. Nếu since_ts=None -> không lọc theo thời gian."""
    base = (base_prefix or "").lower()
    try:
        names = os.listdir(dirpath)
    except FileNotFoundError:
        return []
    paths = []
    for n in names:
        if not n.lower().startswith(base):
            continue
        p = os.path.join(dirpath, n)
        try:
            if since_ts is None or os.path.getmtime(p) >= since_ts - 2:  # buffer 2s
                paths.append(p)
        except FileNotFoundError:
            pass
    return paths


def wait_download_minute_ticks(dirpath, expected_prefix, since_ts, tick_seconds=60, max_minutes=30):
    """
    Mỗi tick 1 phút: kiểm tra tiến độ trong thư mục tải.
    Ưu tiên file mới hơn since_ts; nếu không có thì fallback sang *mọi* file cùng prefix.
    """
    EXT_DONE = (".csv", ".zip", ".xlsx")
    total_ticks = max(1, int(max_minutes * 60 / tick_seconds))
    last_size = None

    def _human_size(n):
        try:
            n = float(n)
        except Exception:
            return "?"
        for unit in ("B","KB","MB","GB","TB"):
            if n < 1024.0:
                return f"{n:.1f}{unit}"
            n /= 1024.0
        return f"{n:.1f}PB"

    print(f"[MONITOR] Waiting up to {max_minutes} minutes, checking every {tick_seconds}s...")

    # Fast path: nếu đã có sẵn file hoàn tất (dù cũ hơn since_ts) -> dùng luôn
    any_paths = _list_matching_files(dirpath, expected_prefix, since_ts=None)
    any_done = [p for p in any_paths if not p.endswith(".crdownload")
                and os.path.splitext(p)[1].lower() in EXT_DONE]
    if any_done:
        target = max(any_done, key=lambda p: os.path.getmtime(p))
        print(f"✅ Found existing file: {os.path.basename(target)} ({_human_size(os.path.getsize(target))})")
        return target

    for i in range(1, total_ticks + 1):
        recent = _list_matching_files(dirpath, expected_prefix, since_ts=since_ts)
        any_paths = recent or _list_matching_files(dirpath, expected_prefix, since_ts=None)  # fallback

        inprog = [p for p in any_paths if p.endswith(".crdownload")]
        done = [p for p in any_paths if not p.endswith(".crdownload")
                and os.path.splitext(p)[1].lower() in EXT_DONE]

        if done and not inprog:
            target = max(done, key=lambda p: os.path.getmtime(p))
            s1 = os.path.getsize(target); time.sleep(2); s2 = os.path.getsize(target)
            if s1 == s2:
                print(f"✅ Download finished: {os.path.basename(target)} ({_human_size(s2)})")
                return target

        if inprog:
            cur = max(inprog, key=lambda p: os.path.getmtime(p))
            try:
                sz = os.path.getsize(cur)
            except FileNotFoundError:
                sz = None
            if sz is not None:
                if last_size is not None and sz > last_size:
                    delta = sz - last_size
                    print(f"[{i:02d}/{total_ticks}] {_human_size(sz)} downloaded (+{_human_size(delta)} since last check)")
                else:
                    print(f"[{i:02d}/{total_ticks}] {_human_size(sz)} downloaded (no change)")
                last_size = sz
        else:
            if not any_paths:
                print(f"[{i:02d}/{total_ticks}] No matching files yet...")
            else:
                names = ", ".join(os.path.basename(p) for p in any_paths)
                print(f"[{i:02d}/{total_ticks}] Files present: {names} (waiting for completion)")

        if i < total_ticks:
            time.sleep(tick_seconds)

    return None

# ---------- Main ----------
def main():
    driver = build_driver()
    try:
        check_and_login(driver, expedia_url, PAGE_LOAD_TIMEOUT)
        open_download_modal(driver)
        set_download_format_to_csv(driver)
        set_export_filename(driver, NAME_PREFIX, PERIOD)
        select_unformatted_and_all_results(driver)

        # 1) Click Download và ghi lại thời điểm
        click_ts = click_modal_download(driver)

        # 2) Theo dõi mỗi 1 phút, tối đa 30 phút
        expected_prefix = f"{NAME_PREFIX} {PERIOD}"
        path = wait_download_minute_ticks(
            DOWNLOAD_DIR,
            expected_prefix,
            since_ts=click_ts,
            tick_seconds=TICK_SECONDS,
            max_minutes=MAX_WAIT_MINUTES,
        )
        if path:
            print(f"✅ Done. File saved at: {path}")
        else:
            print("⏰ Timeout after 30 minutes without finishing download.")
    finally:
        print("Closing Chrome.")
        driver.quit()
if __name__ == "__main__":
    main()

Using local ChromeDriver path.


SessionNotCreatedException: Message: session not created: This version of ChromeDriver only supports Chrome version 145
Current browser version is 147.0.7727.56 with binary path C:\Program Files\Google\Chrome\Application\chrome.exe
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff751ebaa55
	0x7ff751c18630
	0x7ff7519ad75d
	0x7ff7519f5a3d
	0x7ff7519f4785
	0x7ff7519eecd1
	0x7ff7519e95d9
	0x7ff751a3fc51
	0x7ff751a3f4b6
	0x7ff7519f9098
	0x7ff7519f9f83
	0x7ff751ee7810
	0x7ff751ee1afd
	0x7ff751f02c1a
	0x7ff751c33345
	0x7ff751c3b81c
	0x7ff751c21924
	0x7ff751c21ad6
	0x7ff751c07e47
	0x7ff943dae8d7
	0x7ff9440ec3fc
